In [3]:
import pandas as pd
import numpy as np

print("Đang nạp dữ liệu từ thư mục Processed...")
df = pd.read_csv('../data/processed/cleaned_cicids2017.csv', low_memory=False)

# Chuyển lại cột timestamp về định dạng datetime (vì khi lưu ra CSV nó bị biến thành chữ)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

print(f"Nạp thành công! Kích thước dữ liệu: {df.shape}")

Đang nạp dữ liệu từ thư mục Processed...
Nạp thành công! Kích thước dữ liệu: (783678, 88)


In [4]:
import pandas as pd
import numpy as np

print("Đang tạo các bảng Dimension...")

# ==========================================
# 1. Tạo Dimension Label (dim_label)
# ==========================================
dim_label = df[['attack_group']].drop_duplicates().reset_index(drop=True)
dim_label['label_key'] = dim_label.index + 1  # Tạo ID tự tăng
dim_label['is_attack'] = dim_label['attack_group'] != 'Benign'
# Đổi tên cho khớp với schema SQL
dim_label.rename(columns={'attack_group': 'label_raw'}, inplace=True)
dim_label['attack_group'] = dim_label['label_raw'] # Tạm thời để raw và group giống nhau

# ==========================================
# 2. Tạo Dimension Protocol (dim_protocol)
# ==========================================
protocol_col = next((col for col in df.columns if 'protocol' in col.lower()), None)
dim_protocol = df[[protocol_col]].drop_duplicates().reset_index(drop=True)
dim_protocol['protocol_key'] = dim_protocol.index + 1
# Ánh xạ tên giao thức
protocol_map = {6: 'TCP', 17: 'UDP', 0: 'HOPOPT/IPv6', 1: 'ICMP'}
dim_protocol['protocol_name'] = dim_protocol[protocol_col].map(lambda x: protocol_map.get(x, 'Unknown'))
dim_protocol.rename(columns={protocol_col: 'protocol_number'}, inplace=True)

# ==========================================
# 3. Tạo Dimension IP (dim_ip)
# ==========================================
# Gom tất cả IP từ nguồn và đích lại để tạo danh sách IP độc nhất
all_ips = pd.concat([df['source_ip'], df['destination_ip']]).drop_duplicates().reset_index(drop=True)
dim_ip = pd.DataFrame({'ip_address': all_ips})
dim_ip['ip_key'] = dim_ip.index + 1

# Đánh dấu IP nội bộ
import ipaddress
def is_private_ip(ip_str):
    try:
        return ipaddress.ip_address(ip_str).is_private
    except ValueError:
        return False
dim_ip['is_internal'] = dim_ip['ip_address'].apply(is_private_ip)
dim_ip['is_blacklisted'] = False # Tạm đặt False, sau này có file FireHOL sẽ update vào đây

# ==========================================
# 4. Tạo Dimension Time (dim_time)
# ==========================================
dim_time = df[['timestamp']].drop_duplicates().reset_index(drop=True)
# Tạo time_key dạng số nguyên YYYYMMDDHHMM (vd: 201707040830)
dim_time['time_key'] = dim_time['timestamp'].dt.strftime('%Y%m%d%H%M').astype(int)
dim_time['full_datetime'] = dim_time['timestamp']
dim_time['date_value'] = dim_time['timestamp'].dt.date
dim_time['year_num'] = dim_time['timestamp'].dt.year
dim_time['month_num'] = dim_time['timestamp'].dt.month
dim_time['day_num'] = dim_time['timestamp'].dt.day
dim_time['day_name'] = dim_time['timestamp'].dt.day_name()
dim_time['hour_num'] = dim_time['timestamp'].dt.hour
dim_time['minute_num'] = dim_time['timestamp'].dt.minute

# Xác định giờ hành chính (8h - 17h, Thứ 2 - Thứ 6)
dim_time['is_business_hours'] = dim_time['timestamp'].dt.hour.between(8, 17) & (dim_time['timestamp'].dt.dayofweek < 5)

print(f"Hoàn thành Dimension: dim_ip ({len(dim_ip)} dòng), dim_time ({len(dim_time)} dòng)")

Đang tạo các bảng Dimension...
Hoàn thành Dimension: dim_ip (13240 dòng), dim_time (17649 dòng)


In [5]:
print("Đang tạo bảng Fact Flow (Thay thế chữ bằng ID)...")

# Copy dữ liệu gốc để map ID
fact_flow = df.copy()

# 1. Map Time Key
fact_flow['time_key'] = fact_flow['timestamp'].dt.strftime('%Y%m%d%H%M').astype(int)

# 2. Map Protocol Key
fact_flow = fact_flow.merge(dim_protocol, left_on=protocol_col, right_on='protocol_number', how='left')

# 3. Map Label Key
fact_flow = fact_flow.merge(dim_label, left_on='attack_group', right_on='label_raw', how='left')

# 4. Map Source IP Key và Destination IP Key
fact_flow = fact_flow.merge(dim_ip[['ip_address', 'ip_key']], left_on='source_ip', right_on='ip_address', how='left')
fact_flow.rename(columns={'ip_key': 'src_ip_key'}, inplace=True)

fact_flow = fact_flow.merge(dim_ip[['ip_address', 'ip_key']], left_on='destination_ip', right_on='ip_address', how='left')
fact_flow.rename(columns={'ip_key': 'dst_ip_key'}, inplace=True)

# 5. Chọn lọc lại các cột giữ lại cho bảng Fact (Chỉ giữ ID và số đo định lượng)
# Tìm tự động cột byte
byte_fwd_col = next((col for col in df.columns if 'length' in col and 'fwd' in col), None)
byte_bwd_col = next((col for col in df.columns if 'length' in col and 'bwd' in col), None)

fact_columns = [
    'time_key', 'src_ip_key', 'dst_ip_key', 'protocol_key', 'label_key',
    'destination_port', 'flow_duration', byte_fwd_col, byte_bwd_col
]
# Lọc bỏ những cột không tồn tại phòng hờ
fact_columns = [col for col in fact_columns if col in fact_flow.columns]

fact_flow_final = fact_flow[fact_columns]

print(f"Hoàn thành Fact Table: {fact_flow_final.shape}")
display(fact_flow_final.head())

Đang tạo bảng Fact Flow (Thay thế chữ bằng ID)...
Hoàn thành Fact Table: (783678, 9)


,time_key,src_ip_key,dst_ip_key,protocol_key,label_key,destination_port,flow_duration,total_length_of_fwd_packets,total_length_of_bwd_packets
0,201707070353,1,9,1,1,53.0,269.0,64.0,258.0
1,201707070356,2,297,2,1,443.0,112044163.0,2186.0,1937.0
2,201707070401,3,278,2,1,22004.0,10086142.0,6.0,30.0
3,201707070400,4,5,2,1,58028.0,304031.0,12.0,43.0
4,201707070404,5,9,1,1,53.0,991.0,66.0,140.0


In [11]:
import pandas as pd
from sqlalchemy import create_engine, text
import ipaddress

print("1. Đang nạp dữ liệu sạch từ thư mục processed (định dạng CSV)...")
# Đọc file CSV thay vì Parquet
df = pd.read_csv('../data/processed/cleaned_cicids2017.csv', low_memory=False)

# Quan trọng: Ép kiểu lại cột thời gian vì CSV lưu ở dạng chữ
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

print("2. Đang tạo các bảng Dimension bằng Pandas...")
# -- dim_label --
dim_label = df[['attack_group']].drop_duplicates().reset_index(drop=True)
dim_label['label_key'] = dim_label.index + 1
dim_label['is_attack'] = dim_label['attack_group'] != 'Benign'
dim_label.rename(columns={'attack_group': 'label_raw'}, inplace=True)
dim_label['attack_group'] = dim_label['label_raw']

# -- dim_protocol --
protocol_col = next((col for col in df.columns if 'protocol' in col.lower()), None)
dim_protocol = df[[protocol_col]].drop_duplicates().reset_index(drop=True)
dim_protocol['protocol_key'] = dim_protocol.index + 1
protocol_map = {6: 'TCP', 17: 'UDP', 0: 'HOPOPT/IPv6', 1: 'ICMP'}
dim_protocol['protocol_name'] = dim_protocol[protocol_col].map(lambda x: protocol_map.get(x, 'Unknown'))
dim_protocol.rename(columns={protocol_col: 'protocol_number'}, inplace=True)

# -- dim_ip --
all_ips = pd.concat([df['source_ip'], df['destination_ip']]).drop_duplicates().reset_index(drop=True)
dim_ip = pd.DataFrame({'ip_address': all_ips})
dim_ip['ip_key'] = dim_ip.index + 1
dim_ip['is_internal'] = dim_ip['ip_address'].apply(lambda x: ipaddress.ip_address(str(x)).is_private if pd.notnull(x) else False)
dim_ip['is_blacklisted'] = False 

# -- dim_time (Chống trùng lặp Key) --
df['timestamp_rounded'] = df['timestamp'].dt.floor('min')
dim_time = df[['timestamp_rounded']].drop_duplicates().reset_index(drop=True)
dim_time['time_key'] = dim_time['timestamp_rounded'].dt.strftime('%Y%m%d%H%M').astype('int64')
dim_time['full_datetime'] = dim_time['timestamp_rounded']
dim_time['date_value'] = dim_time['timestamp_rounded'].dt.date
dim_time['year_num'] = dim_time['timestamp_rounded'].dt.year
dim_time['month_num'] = dim_time['timestamp_rounded'].dt.month
dim_time['day_num'] = dim_time['timestamp_rounded'].dt.day
dim_time['day_name'] = dim_time['timestamp_rounded'].dt.day_name()
dim_time['hour_num'] = dim_time['timestamp_rounded'].dt.hour
dim_time['minute_num'] = dim_time['timestamp_rounded'].dt.minute
dim_time['is_business_hours'] = dim_time['timestamp_rounded'].dt.hour.between(8, 17) & (dim_time['timestamp_rounded'].dt.dayofweek < 5)
dim_time = dim_time.drop_duplicates(subset=['time_key'], keep='first')

print("3. Đang tạo bảng Fact Flow...")
fact_flow = df.copy()
fact_flow['time_key'] = fact_flow['timestamp'].dt.floor('min').dt.strftime('%Y%m%d%H%M').astype('int64')
fact_flow = fact_flow.merge(dim_protocol, left_on=protocol_col, right_on='protocol_number', how='left')
fact_flow = fact_flow.merge(dim_label, left_on='attack_group', right_on='label_raw', how='left')
fact_flow = fact_flow.merge(dim_ip[['ip_address', 'ip_key']], left_on='source_ip', right_on='ip_address', how='left').rename(columns={'ip_key': 'src_ip_key'})
fact_flow = fact_flow.merge(dim_ip[['ip_address', 'ip_key']], left_on='destination_ip', right_on='ip_address', how='left').rename(columns={'ip_key': 'dst_ip_key'})

byte_fwd_col = next((col for col in df.columns if 'length' in col and 'fwd' in col), None)
byte_bwd_col = next((col for col in df.columns if 'length' in col and 'bwd' in col), None)

fact_columns = ['time_key', 'src_ip_key', 'dst_ip_key', 'protocol_key', 'label_key', 'destination_port', 'flow_duration', byte_fwd_col, byte_bwd_col]
fact_columns = [col for col in fact_columns if col in fact_flow.columns]
fact_flow_final = fact_flow[fact_columns]

print("4. Đang kết nối và đẩy dữ liệu vào SQL Server...")
# Thay tên server khớp với lúc nãy (nhớ để chữ r ở trước)
server_name = r'localhost\SQLEXPRESS' 
db_name = 'cicids2017_dw'

conn_str = f"mssql+pyodbc://@{server_name}/{db_name}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(conn_str, fast_executemany=True)

try:
    with engine.begin() as conn:
        print(" -> Đang dọn dẹp dữ liệu cũ (để tránh lỗi trùng lặp)...")
        conn.execute(text("DELETE FROM fact_flow"))
        conn.execute(text("DELETE FROM dim_time"))
        conn.execute(text("DELETE FROM dim_ip"))
        conn.execute(text("DELETE FROM dim_label"))
        conn.execute(text("DELETE FROM dim_protocol"))

        print(" -> Nạp dim_protocol...")
        conn.execute(text("SET IDENTITY_INSERT dim_protocol ON"))
        dim_protocol[['protocol_key', 'protocol_number', 'protocol_name']].to_sql('dim_protocol', conn, if_exists='append', index=False)
        conn.execute(text("SET IDENTITY_INSERT dim_protocol OFF"))

        print(" -> Nạp dim_label...")
        conn.execute(text("SET IDENTITY_INSERT dim_label ON"))
        dim_label[['label_key', 'label_raw', 'attack_group', 'is_attack']].to_sql('dim_label', conn, if_exists='append', index=False)
        conn.execute(text("SET IDENTITY_INSERT dim_label OFF"))

        print(" -> Nạp dim_ip...")
        conn.execute(text("SET IDENTITY_INSERT dim_ip ON"))
        dim_ip[['ip_key', 'ip_address', 'is_internal', 'is_blacklisted']].to_sql('dim_ip', conn, if_exists='append', index=False)
        conn.execute(text("SET IDENTITY_INSERT dim_ip OFF"))

        print(" -> Nạp dim_time...")
        dim_time_to_load = dim_time.drop(columns=['timestamp_rounded'])
        dim_time_to_load.to_sql('dim_time', conn, if_exists='append', index=False)
        
        print(" -> Nạp fact_flow...")
        fact_flow_final.to_sql('fact_flow', conn, if_exists='append', index=False)
        
    print("✅ XUẤT SẮC! Đã nạp thành công toàn bộ dữ liệu vào Data Warehouse!")
except Exception as e:
    print(f"❌ Có lỗi xảy ra trong quá trình nạp: {e}")

1. Đang nạp dữ liệu sạch từ thư mục processed (định dạng CSV)...
2. Đang tạo các bảng Dimension bằng Pandas...
3. Đang tạo bảng Fact Flow...
4. Đang kết nối và đẩy dữ liệu vào SQL Server...
 -> Đang dọn dẹp dữ liệu cũ (để tránh lỗi trùng lặp)...
 -> Nạp dim_protocol...
 -> Nạp dim_label...
 -> Nạp dim_ip...
 -> Nạp dim_time...
 -> Nạp fact_flow...
✅ XUẤT SẮC! Đã nạp thành công toàn bộ dữ liệu vào Data Warehouse!


In [ ]:
# ==============================================================================
# NOTEBOOK 02: THỰC THI VÀ KIỂM TRA QUÁ TRÌNH ETL (POSTGRESQL)
# ==============================================================================
import os
import sys

# 1. Chỉ đường cho Python lùi ra ngoài thư mục gốc để nhận diện thư mục 'src'
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append(os.getcwd())

print("🚀 BƯỚC 1: KÍCH HOẠT HỆ THỐNG ETL TỰ ĐỘNG...")
# Gọi lệnh chạy pipeline kèm cờ tạo schema (Nếu đã tạo schema rồi thì bỏ cờ --create-schema đi)
!python -m src.etl.run_etl --create-schema

print("\n🔍 BƯỚC 2: KIỂM TRA TÍNH TOÀN VẸN CỦA KHO DỮ LIỆU POSTGRESQL...")
# Tái sử dụng hàm kết nối chuẩn từ dự án
from src.utils.config import get_connection

try:
    # Kết nối sẽ tự động lấy từ file .env
    conn = get_connection()
    cur = conn.cursor()
    
    print("📊 BÁO CÁO SỐ LƯỢNG BẢN GHI TRONG CÁC BẢNG (STAR SCHEMA):")
    # Danh sách các bảng đã được bổ sung thêm dim_port
    tables = ['dim_protocol', 'dim_label', 'dim_port', 'dim_ip', 'dim_time', 'fact_flow']
    
    for table in tables:
        cur.execute(f"SELECT COUNT(*) FROM {table}")
        count = cur.fetchone()[0]
        print(f"   + Bảng {table:<15}: {count:>10,} dòng")
        
    print("-" * 50)
    print("✅ KẾT LUẬN:")
    print(" - Toàn bộ dữ liệu đã nạp thành công vào PostgreSQL.")
    print(" - Sẵn sàng tiến tới bước truy vấn OLAP và Data Mining.")
    
    cur.close()
    conn.close()
except Exception as e:
    print(f"❌ Lỗi khi kiểm tra dữ liệu: {e}")